In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit


# Importing from your provided scripts
from stage1 import lasso_rolling_window_fast, to_ar1_innovations
from stage2 import compute_stage2_r_squared

# Set a random seed for reproducibility
np.random.seed(42)

The standard deviation of the return series is itself endogenous and depends on the varaince of the topics sereis and the lasso parameter lambda. 
We could generate the full return series, which means that kappa is not related to the returns, they are not a product of the ALM.
Alternativley, we can run the model once, look at the standard deviations of returns, and then rerun the model but add as much noise in the second stage as we need to bring the return se close to what we want.
The function calibrate_alm_variance_empirical() does exactly this. 

What remains to do:

    - think about whether this makes sense
    - code a loop that tests a range of standard deviations and stores them along with n_active and r2

In [98]:
# Parameter calibration
a = 1.001 # mean of the dividend process
gamma = 2.0 # risk aversion
sigma = 0.01 # std dev of the dividend process
delta = 0.96 # discount factor
phi = np.exp(sigma**2 / 2) # correction term for lognormality
kappa = delta * a ** ( - gamma) * phi
w = - np.log(kappa) - np.exp(-10)  # small constant to ensure positivity
print(f"w: {w}, kappa: {kappa}")

w: 0.04272559525665953, kappa: 0.9581307815062262


In [99]:
# load topics
topics = pd.read_csv(r"C:\Users\jonat\Lasso_paper\Empirical\data\merged_return_topic_data.csv")
# get another df only with stocks columns, those with numbers in names
stocks = topics[[col for col in topics.columns if str(col).replace('.', '', 1).isdigit()]]
# keep only the relevant columns (those that conatin no numbers in their name)
topics = topics[[col for col in topics.columns if not any(char.isdigit() for char in col)]]
# set date as index
topics.set_index('date', inplace=True)
# extract innovations
topics = to_ar1_innovations(topics)
# delete first row since it will be all NaN after the AR(1) transformation
topics = topics.iloc[1:]

In [100]:
from sklearn.discriminant_analysis import StandardScaler


def multivariate_CGL_offline_lasso(X, sigma, kappa, w, rolling_window_size=30, lambda_lasso=0.01, correct_with_tanh=True, random_state=42):
        T, K = X.shape
        assert rolling_window_size < T, "rolling_window_size must be less than T"
        x = X.values
        beta_offline = np.zeros((T, K, 1))
        r_offline = np.zeros(T)
        n_active = 0  # Initialize the counter

        if random_state is not None:
            np.random.seed(random_state)

        r_offline[1:rolling_window_size+1] = np.random.normal(0, sigma, rolling_window_size)

        for t in range(rolling_window_size+1, T):
            x_current = x[t-rolling_window_size-1:t-1, :] 
            r_offline_current = r_offline[t-rolling_window_size:t]
            scaler = StandardScaler()
            x_current_scaled = scaler.fit_transform(x_current)
            
            # Fit Lasso and store coefficients
            lasso_model = Lasso(alpha=lambda_lasso, fit_intercept=False, random_state=random_state, max_iter=2000).fit(x_current_scaled, r_offline_current)
            beta_offline[t] = lasso_model.coef_.reshape(-1,1)
            
            # NEW LINE: Increment n_active by the number of non-zero coefficients in the current window
            n_active += np.count_nonzero(beta_offline[t])
            
            if correct_with_tanh:
                r_offline[t] = (np.log(1 - kappa * np.exp(w * np.tanh((beta_offline[t-1, :, 0] @ x[t-1, :].reshape(-1, 1)) / w))) - np.log(1 - kappa * np.exp(w * np.tanh((beta_offline[t, :, 0] @ x[t, :].reshape(-1, 1)) / w))) + np.random.normal(0, sigma)).item()
            else:
                r_offline[t] = (np.log(1 - kappa * np.exp((beta_offline[t-1, :, 0] @ x[t-1, :].reshape(-1, 1)))) - np.log(1 - kappa * np.exp((beta_offline[t, :, 0] @ x[t, :].reshape(-1, 1)))) + np.random.normal(0, sigma)).item()
        
        return beta_offline, r_offline, n_active


beta_offline, r_offline, n_active = multivariate_CGL_offline_lasso(topics, sigma, kappa, w, rolling_window_size = 100, lambda_lasso =  0.00001, correct_with_tanh=True, random_state=42)

print(f"Total number of active coefficients across all time periods: {n_active}")

# get standard deviation of r_offline
r_offline_std = np.std(r_offline)
print(f"Standard deviation of r_offline: {r_offline_std}")
# print median standard deviation of each topic
print("Median standard deviation of each topic:")
print(np.median(topic_stds))

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.839e-07, tolerance: 8.273e-07
  model = cd_fast.enet_coordinate_descent(
c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.757e-07, tolerance: 8.451e-07
  model = cd_fast.enet_coordinate_descent(
c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

KeyboardInterrupt: 

In [ ]:
import numpy as np
from scipy.optimize import least_squares
from sklearn.metrics import r2_score

def estimate_kappa_nls(X, beta_offline, r_offline, w, correct_with_tanh=True, rolling_window_size=30):
    """
    Estimates the parameter kappa using Non-linear Least Squares, 
    given a fixed value for w, and computes the R2 of the predicted values.
    """
    # Ensure X is a numpy array
    x_vals = X.values if hasattr(X, 'values') else X
    start_t = rolling_window_size + 1
    
    # 1. Compute dot products Z_t = beta_t^T * x_t for all time steps
    Z = np.sum(beta_offline[:, :, 0] * x_vals, axis=1)
    
    # Extract t-1 and t arrays matching the r_offline loop scope
    Z_prev = Z[start_t-1:-1]
    Z_curr = Z[start_t:]
    
    # 2. Compute exponential terms based on the correction flag
    if correct_with_tanh:
        term_prev = np.exp(w * np.tanh(Z_prev / w))
        term_curr = np.exp(w * np.tanh(Z_curr / w))
    else:
        term_prev = np.exp(Z_prev)
        term_curr = np.exp(Z_curr)
        
    # 3. Define the residual function to minimize
    def residuals(kappa_arr):
        k = kappa_arr[0]
        val_prev = 1 - k * term_prev
        val_curr = 1 - k * term_curr
        
        # Heavy penalty if kappa causes a non-positive log argument
        if np.any(val_prev <= 0) or np.any(val_curr <= 0):
            return np.full_like(r_offline[start_t:], 1e6)
            
        r_pred = np.log(val_prev) - np.log(val_curr)
        return r_offline[start_t:] - r_pred

    # 4. Set upper bound to guarantee valid logarithms (1 - kappa * exp(...) > 0)
    max_term = max(np.max(term_prev), np.max(term_curr))
    upper_bound = 0.9999 / max_term
    
    # 5. Execute NLS Optimization
    res = least_squares(residuals, x0=[0.0], bounds=(-np.inf, upper_bound))
    kappa_est = res.x[0]
    
    # 6. Compute R-squared on the valid time window
    r_pred_final = np.log(1 - kappa_est * term_prev) - np.log(1 - kappa_est * term_curr)
    r2 = r2_score(r_offline[start_t:], r_pred_final)
    
    return kappa_est, r2